# Visualize channel z-sweep: side-by-side GIF + mean-intensity-vs-z plot

Standalone diagnostic: given **one image file** (`.dax`, `.tiff`/`.tif`, or `.zarr` --
default `.zarr`) and its **frame table** CSV, builds

1. An animated **GIF**, one panel per channel side by side, sweeping from the
   shallowest to the deepest z present in the frame table.
2. A **mean intensity vs. z** line plot, one line per channel.

Unlike `create_mosaic_gif.ipynb` (many FOVs tiled by stage position, one
channel, needs a full `SAMPLE_DIR`/`round_info.csv`/`ExperimentMetadata`
context), this notebook needs only the image file + frame-table path -- no
experiment metadata at all, so it works on any single file (a quick look at
one FOV, a test acquisition, a file pulled off the microscope for a sanity
check, ...).

Follows [`NOTEBOOK_GUIDELINES.md`](../../NOTEBOOK_GUIDELINES.md): calculation
cells are separated from display cells, frame reads are cached (under
`OUTPUT_DIR/cache/`) so re-running to tweak GIF/plot parameters doesn't
re-read from disk, and `ProgressReporter` reports progress on the read loop.
For a single file (tens to a few hundred frames, one read pass) this is
normally quick -- no `USE_SLURM_ARRAY` option here, unlike the heavy
multi-FOV notebooks, since there's no batch of files to parallelize over.

## 1 — Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from IPython.display import Image as IPyImage, display

MERCI_DIR = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.io        import iter_image_frames
from MERci.progress_display import ProgressReporter

print(f"MERCI_DIR: {MERCI_DIR}")

## 2 — Parameters

In [ ]:
# Image file to sweep -- .dax, .tiff/.tif, or .zarr (default; must be whichever
# format HAL actually wrote).
IMAGE_PATH = Path(r"path/to/fov.zarr")

# Frame table CSV matching this exact image (columns: color [nm, NaN=blank],
# channel [hardware index], z [um]; frame-number index) -- e.g.
# metadata/frame-table-bits-<name>.csv or metadata/frame-table-cells-<name>.csv.
FRAME_TABLE_PATH = Path(r"path/to/frame-table-bits-<name>.csv")

# Only used for .dax (raw binary with no embedded shape) -- ignored for .zarr/.tiff.
FRAME_WIDTH  = None
FRAME_HEIGHT = None

# Where the GIF/plot/cache are written -- defaults to a folder next to the image file.
OUTPUT_DIR = IMAGE_PATH.parent / "zsweep_diagnostics"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- GIF parameters --------------------------------------------------------
GIF_Z_STRIDE             = 1      # every Nth z-step -- 1 = every step, higher = coarser/faster preview
GIF_FRAME_DURATION_MS    = 300    # per-frame display duration in the saved GIF
CONTRAST_PERCENTILE_CLIP = (1.0, 99.0)   # contrast stretch, computed PER CHANNEL (section 5)
PANEL_WIDTH_PX           = 300    # each channel panel is resized to this width before compositing
PANEL_PADDING_PX         = 8

# nm -> display colour, for both the GIF panel labels and the intensity-plot lines
# (same mapping as MERci.visualization.visualize_shutter_sequence's _WAVELENGTH_COLOUR).
CHANNEL_COLOR_HEX = {405: "#9467bd", 488: "#1f77b4", 560: "#ff7f0e", 650: "#2ca02c", 750: "#d62728"}
DEFAULT_COLOR_HEX = "#7f7f7f"

# Explicit plot font sizes (NOTEBOOK_GUIDELINES.md #5).
PLOT_TITLE_FONTSIZE  = 14
PLOT_LABEL_FONTSIZE  = 12
PLOT_TICK_FONTSIZE   = 11
PLOT_LEGEND_FONTSIZE = 10

print(f"Image       : {IMAGE_PATH}")
print(f"Frame table : {FRAME_TABLE_PATH}")
print(f"Output dir  : {OUTPUT_DIR}")

## 3 — Load the frame table and group frames by channel

In [ ]:
if not IMAGE_PATH.exists():
    raise FileNotFoundError(f"Image not found: {IMAGE_PATH}")
if not FRAME_TABLE_PATH.exists():
    raise FileNotFoundError(f"Frame table not found: {FRAME_TABLE_PATH}")

frame_table = pd.read_csv(FRAME_TABLE_PATH, index_col=0)

# Only real imaging frames (a non-blank colour) -- blank/return-to-bead-z frames
# have color=NaN and carry no channel image worth showing.
imaging_frames = frame_table[frame_table["color"].notna()].copy()
if imaging_frames.empty:
    raise ValueError(f"No non-blank (color) frames found in {FRAME_TABLE_PATH}")

# Round to guard against float noise in the stored color value (same convention
# used throughout this codebase, e.g. acquisition/configs.py's channel lookups).
imaging_frames["color"] = imaging_frames["color"].round(1)
channels = sorted(imaging_frames["color"].unique())
frames_by_channel = {
    color: imaging_frames[imaging_frames["color"] == color].sort_values("z")
    for color in channels
}
z_positions = sorted(imaging_frames["z"].unique())

print(f"{len(channels)} channel(s): {channels}")
for color in channels:
    sub = frames_by_channel[color]
    print(f"  {color:6.1f} nm: {len(sub)} frame(s), z {sub['z'].min():.1f}-{sub['z'].max():.1f} um")
print(f"z sweep (all channels pooled): {len(z_positions)} step(s), "
      f"{min(z_positions):.1f}-{max(z_positions):.1f} um")

## 4 — Read every needed frame (cached)

Cached to `OUTPUT_DIR/cache/frames/frame{idx}.npy`, one file per frame index --
re-running this notebook to tweak `CONTRAST_PERCENTILE_CLIP`/GIF parameters
never re-reads from disk.

In [ ]:
# ---- Calculation ----------------------------------------------------------
frames_cache_dir = OUTPUT_DIR / "cache" / "frames"
frames_cache_dir.mkdir(parents=True, exist_ok=True)


def frame_cache_path(frame_idx):
    return frames_cache_dir / f"frame{int(frame_idx):04d}.npy"


all_frame_indices = sorted(int(i) for i in imaging_frames.index.tolist())
to_read = [idx for idx in all_frame_indices if not frame_cache_path(idx).exists()]
print(f"{len(all_frame_indices) - len(to_read)} frame(s) already cached; {len(to_read)} to read.")

if to_read:
    reporter = ProgressReporter(total=len(to_read), label="Reading frames")
    for frame_idx, frame in reporter.wrap(iter_image_frames(
        IMAGE_PATH, to_read, frame_width=FRAME_WIDTH, frame_height=FRAME_HEIGHT,
    )):
        np.save(frame_cache_path(frame_idx), frame)

frames = {idx: np.load(frame_cache_path(idx)) for idx in all_frame_indices}
print(f"Loaded {len(frames)} frame(s) into memory.")

## 5 — Side-by-side GIF (one panel per channel, animated over z)

**Calculation cell**, then **display cell** (`NOTEBOOK_GUIDELINES.md` #1).

Contrast is stretched **per channel** (not one shared scale across all
channels) -- different lasers/dyes have very different intensity ranges, so a
single shared scale would wash out the dimmer channels.

For each z-step, every channel shows its own frame at the largest z **at or
below** that step (floor, not nearest -- same convention used elsewhere in
this codebase, e.g. `04_measure_tissue_thickness.ipynb`'s `floor_z_position`,
so a panel never shows content from a z that hasn't actually been reached
yet). A channel imaged at only a single z (e.g. a bead/DAPI reference frame)
simply repeats that one frame for every step.

In [ ]:
# ---- Calculation ----------------------------------------------------------
lo_pct, hi_pct = CONTRAST_PERCENTILE_CLIP
channel_vrange = {}
for color in channels:
    pooled = np.concatenate([frames[int(idx)].ravel() for idx in frames_by_channel[color].index])
    channel_vrange[color] = tuple(np.percentile(pooled, [lo_pct, hi_pct]))
    vmin, vmax = channel_vrange[color]
    print(f"  {color:6.1f} nm contrast range (p{lo_pct:.0f}-p{hi_pct:.0f}): [{vmin:.0f}, {vmax:.0f}]")


def floor_frame_index(color, z_target):
    """Index of channel `color`'s own frame at the largest z <= z_target
    (floor, not nearest). Falls back to that channel's shallowest frame if
    z_target is below every z it was actually imaged at."""
    sub = frames_by_channel[color]   # already sorted by z (section 3)
    z_vals = sub["z"].to_numpy()
    idx_vals = sub.index.to_numpy()
    valid = np.where(z_vals <= z_target)[0]
    pos = int(valid[-1]) if len(valid) else 0
    return int(idx_vals[pos])


gif_z_positions = z_positions[::GIF_Z_STRIDE]
print(f"GIF_Z_STRIDE={GIF_Z_STRIDE} -> {len(gif_z_positions)} step(s) "
      f"(of {len(z_positions)} available), {len(channels)} channel(s) per step.")

In [ ]:
# ---- Display ----------------------------------------------------------------
def to_uint8(frame, vmin, vmax):
    scaled = (frame.astype(np.float64) - vmin) / max(vmax - vmin, 1e-9) * 255
    return np.clip(scaled, 0, 255).astype(np.uint8)


def compose_side_by_side(panels, labels, z_um):
    """One RGB PIL image: `panels` (list of uint8 2D arrays) resized to
    PANEL_WIDTH_PX and pasted left-to-right with PANEL_PADDING_PX gaps, each
    labelled with its channel (in that channel's own display colour) and the
    shared z value."""
    resized = []
    for panel in panels:
        h, w = panel.shape
        scale = PANEL_WIDTH_PX / w
        resized.append(Image.fromarray(panel).convert("RGB").resize((PANEL_WIDTH_PX, max(1, int(h * scale)))))
    panel_h  = max(img.height for img in resized)
    canvas_w = len(resized) * PANEL_WIDTH_PX + (len(resized) + 1) * PANEL_PADDING_PX
    canvas_h = panel_h + 3 * PANEL_PADDING_PX + 20
    canvas   = Image.new("RGB", (canvas_w, canvas_h), color=(0, 0, 0))
    draw     = ImageDraw.Draw(canvas)
    font     = ImageFont.load_default(size=max(14, PANEL_WIDTH_PX // 20))

    x = PANEL_PADDING_PX
    for img, label in zip(resized, labels):
        canvas.paste(img, (x, PANEL_PADDING_PX))
        color_hex = CHANNEL_COLOR_HEX.get(round(label), DEFAULT_COLOR_HEX)
        draw.text((x, PANEL_PADDING_PX + panel_h + 4), f"{label:.0f} nm", fill=color_hex, font=font)
        x += PANEL_WIDTH_PX + PANEL_PADDING_PX
    draw.text((PANEL_PADDING_PX, canvas_h - 20), f"z = {z_um:.1f} um", fill="white", font=font)
    return canvas


pil_frames = []
reporter = ProgressReporter(total=len(gif_z_positions), label="Assembling GIF frames")
for z in reporter.wrap(gif_z_positions):
    panels = [to_uint8(frames[floor_frame_index(color, z)], *channel_vrange[color]) for color in channels]
    pil_frames.append(compose_side_by_side(panels, channels, z))

gif_path = OUTPUT_DIR / f"{IMAGE_PATH.stem}_channel_zsweep.gif"
pil_frames[0].save(gif_path, save_all=True, append_images=pil_frames[1:],
                    duration=GIF_FRAME_DURATION_MS, loop=0)
print(f"Saved: {gif_path}  ({len(pil_frames)} frame(s), {len(channels)} channel(s) side by side)")
display(IPyImage(filename=str(gif_path)))

## 6 — Mean intensity vs. z, one line per channel

**Calculation cell**, then **display cell** (`NOTEBOOK_GUIDELINES.md` #1).

In [ ]:
# ---- Calculation ----------------------------------------------------------
intensity_records = []
for color in channels:
    sub = frames_by_channel[color]
    for frame_idx, z in zip(sub.index, sub["z"]):
        intensity_records.append({
            "channel_nm": color, "z_um": float(z), "mean_intensity": float(frames[int(frame_idx)].mean()),
        })
intensity_df = pd.DataFrame(intensity_records)

intensity_csv = OUTPUT_DIR / f"{IMAGE_PATH.stem}_channel_zsweep_intensity.csv"
intensity_df.to_csv(intensity_csv, index=False)
print(f"Saved: {intensity_csv}")
print(intensity_df.groupby("channel_nm")["mean_intensity"].describe())

In [ ]:
# ---- Display ----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
for color in channels:
    sub = intensity_df[intensity_df["channel_nm"] == color].sort_values("z_um")
    color_hex = CHANNEL_COLOR_HEX.get(round(color), DEFAULT_COLOR_HEX)
    ax.plot(sub["z_um"], sub["mean_intensity"], "-o", color=color_hex, label=f"{color:.0f} nm", markersize=3)
ax.set_xlabel("z (um)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_ylabel("Mean intensity", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title(f"{IMAGE_PATH.name} -- mean intensity vs. z", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
fig.tight_layout()

intensity_fig_path = OUTPUT_DIR / f"{IMAGE_PATH.stem}_channel_zsweep_intensity.png"
fig.savefig(intensity_fig_path, dpi=150)
plt.show()
print(f"Saved: {intensity_fig_path}")